# Day 1 — Statistical Inference Foundations
### Molecular Biology & Clinical Data Analysis Workshop

**Today's goal:** understand what confidence intervals and p-values actually mean, using real clinical trial data — not to become a Python programmer.

---


## How to use this notebook

- You will **not** be asked to write code from scratch today. Every code cell either runs as-is, or has **one clearly marked line** for you to change (look for `# 🔧 CHANGE THIS`).
- Where you see an **🤖 AI Prompt** box, try pasting that prompt into Claude, ChatGPT, or the Gemini panel in Colab (the ✨ spark icon) and compare what it gives you to the pre-written cell below it.
- Where you see a **💬 Discuss** box, pause — these are the moments that matter most today. Talk with your neighbor or the group before moving on.
- If a code cell errors out, don't debug it yourself — just re-run the **Setup** cell below, or ask your facilitator.


## Setup — run this cell first (and any time something looks broken)

In [ ]:
# Run this cell once at the start of the day.
# It installs the one extra package we need and loads the libraries.

!pip install -q lifelines

import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
print("Setup complete — you're ready to go.")


## Section 1 — Meet today's dataset

We're using the **GBSG2** dataset: 686 women with node-positive breast cancer, from a real
randomized trial run by the German Breast Cancer Study Group. It looked at whether hormonal
therapy affected outcomes.

Columns include: hormonal therapy (`horTh`), age, menopausal status, tumor size (`tsize`),
tumor grade (`tgrade`), number of positive lymph nodes (`pnodes`), hormone receptor levels,
follow-up time in days (`time`), and whether an event — recurrence or death — was observed
by the end of follow-up (`cens`, 1 = event observed, 0 = censored/no event yet).


In [ ]:
from lifelines.datasets import load_gbsg2

gbsg2 = load_gbsg2()
print(f"Number of patients: {len(gbsg2)}")
gbsg2.head()


### 💬 Discuss

These 686 women are a **sample**. What's the **population** we'd want to generalize to? Is it
"all women with node-positive breast cancer," or something narrower (e.g. depending on when/where
the trial ran, eligibility criteria, etc.)? Why does that matter before we trust any conclusion
from this data?


## Section 2 — Confidence Intervals

**Concept check (unplugged):** if we say "the 95% CI for the recurrence rate is 35%–42%," what
does that *actually* mean? (Hint: it is **not** "there's a 95% chance the true rate is in this
range." We'll unpack why on the whiteboard before running any code.)

### Exercise 2.1 — CI on a proportion: recurrence rate

We'll compute a confidence interval for the proportion of women who had a recurrence/death event
(`cens == 1`) during the study.


**🤖 AI Prompt to try:**

> "I have a pandas dataframe `gbsg2` with a binary column `cens` (1 = event occurred, 0 = no
> event). Compute the proportion of 1s and a 95% confidence interval for that proportion,
> explain which method you used, and explain in plain language what the interval means."

Compare what the AI gives you to the cell below — does it match? Does its *explanation* of what
the CI means hold up, or does it slip into the "95% chance the true value is in here" mistake?


In [ ]:
# 🔧 CHANGE THIS: try 0.90 or 0.99 to see how the interval width changes
confidence_level = 0.95

n_events = gbsg2['cens'].sum()
n_total = len(gbsg2)
proportion = n_events / n_total

ci_low, ci_high = proportion_confint(count=n_events, nobs=n_total, alpha=1 - confidence_level, method='wilson')

print(f"Recurrence/event rate: {proportion:.1%}")
print(f"{confidence_level:.0%} CI: [{ci_low:.1%}, {ci_high:.1%}]")


### 💬 Discuss

- What happened to the interval when you changed the confidence level? Why?
- If this trial had enrolled 6,860 women instead of 686, what would you expect to happen to the
  interval width — and why does that matter for how much a small pilot study can really tell you?


### Exercise 2.2 — CI on a mean: tumor size

Now let's build a CI for a *continuous* variable instead of a proportion — average tumor size (`tsize`, in mm).


**🤖 AI Prompt to try:**

> "I have a pandas dataframe `gbsg2` with a numeric column `tsize` (tumor size in mm). Compute
> the mean and a 95% confidence interval for the mean, and briefly explain the assumptions
> behind the method you used."


In [ ]:
# 🔧 CHANGE THIS: try a different numeric column, e.g. 'age' or 'pnodes'
variable = 'tsize'

data = gbsg2[variable].dropna()
mean = data.mean()
sem = stats.sem(data)  # standard error of the mean
ci_low, ci_high = stats.t.interval(confidence=0.95, df=len(data)-1, loc=mean, scale=sem)

print(f"Mean {variable}: {mean:.2f}")
print(f"95% CI: [{ci_low:.2f}, {ci_high:.2f}]")


### 💬 Discuss

The AI likely used a method that assumes something about the *shape* of the data (e.g. that the
sampling distribution of the mean is approximately normal). Plot a histogram of `tsize` below —
does the assumption look reasonable? What would you ask the AI to check before trusting its CI?


In [ ]:
# No changes needed — just run this to look at the shape of the data
gbsg2['tsize'].hist(bins=20)
plt.xlabel('Tumor size (mm)')
plt.ylabel('Count')
plt.title('Distribution of tumor size')
plt.show()


## Section 3 — Hypothesis Testing & p-values

**Concept check (unplugged):** before running anything, recall — a p-value is the probability of
seeing data this extreme (or more) **if the null hypothesis were true**. It is *not* the
probability that the null hypothesis is true, and it is *not* the probability the result is "real."

### Exercise 3.1 — Checking baseline balance between trial arms

Before anyone trusts a comparison of *outcomes* between treatment groups, a standard first step
in analyzing any randomized trial is checking **baseline balance**: were the two arms actually
comparable on other measured characteristics before treatment started? If hormonal therapy
patients were, say, meaningfully older or had larger tumors going in, that would be a red flag
about the randomization — not about the treatment itself.

(We're deliberately not touching the actual recurrence/survival outcome yet — comparing
time-to-event data properly requires handling censoring, which is its own topic we'll cover
on Day 3, after regression.)


**🤖 AI Prompt to try:**

> "I have a pandas dataframe `gbsg2` with a numeric column `tsize` (tumor size in mm) and a
> column `horTh` ('yes'/'no', hormonal therapy). Run a two-sample t-test comparing `tsize`
> between the two groups, report the p-value, and explain in plain language what conclusion I
> can and cannot draw from it."


In [ ]:
# 🔧 CHANGE THIS: try 'age' or 'pnodes' to check balance on a different baseline variable
balance_variable = 'tsize'

group_yes = gbsg2[gbsg2['horTh'] == 'yes'][balance_variable]
group_no = gbsg2[gbsg2['horTh'] == 'no'][balance_variable]

t_stat, p_value = stats.ttest_ind(group_yes, group_no)

print(f"Mean {balance_variable} (hormonal therapy): {group_yes.mean():.1f}")
print(f"Mean {balance_variable} (no hormonal therapy): {group_no.mean():.1f}")
print(f"p-value: {p_value:.4f}")


### 💬 Discuss — spot the misinterpretation

Below are three ways someone might describe this result. Which (if any) are correct? Which did
the AI's explanation resemble?

1. *"p = 0.03, so there's a 97% chance the groups really differ in tumor size."*
2. *"p = 0.03, so if the groups truly had no underlying difference, we'd see a gap this large or
   larger only 3% of the time by chance."*
3. *"p = 0.03, so hormonal therapy definitely causes different tumor sizes."*

(Only one of these is defensible — and it's easy to see how an AI assistant, or a rushed reader,
slides into the other two. Also worth noting: statement 3 doesn't even make causal sense here —
tumor size was measured *before* treatment assignment, so treatment can't have caused it. That's
a good reminder to always ask "which came first?")


### Exercise 3.2 — Why a single p-value can mislead you: a quick simulation

Here's the uncomfortable truth: even when there is genuinely **no real effect**, about 5% of the
time a test will still produce p < 0.05, purely by chance. Let's see it happen.


In [ ]:
# 🔧 CHANGE THIS: try 500 or 5000 to see the false-positive rate stabilize
n_simulations = 1000

np.random.seed(0)
false_positives = 0

for _ in range(n_simulations):
    # Two samples drawn from the *same* distribution — there is NO real difference here
    sample_a = np.random.normal(loc=0, scale=1, size=50)
    sample_b = np.random.normal(loc=0, scale=1, size=50)
    _, p = stats.ttest_ind(sample_a, sample_b)
    if p < 0.05:
        false_positives += 1

print(f"Out of {n_simulations} tests where there was truly NO effect,")
print(f"{false_positives} ({false_positives/n_simulations:.1%}) still came back p < 0.05.")


### 💬 Discuss

- Does that percentage look familiar? Why does it hover around 5%?
- Now imagine a genomics experiment testing not 1 hypothesis, but **20,000** genes at once
  (tomorrow's dataset). If each test has a 5% false-positive rate under the null, roughly how many
  "significant" genes would you expect to find *by chance alone*, even if nothing in the genome
  actually changed? We'll pick this up first thing on Day 2.


## Section 3.3 — Type I and Type II Errors

The simulation above showed something important: even with **no real effect**, about 5% of tests
still came back "significant." That's a **Type I error** — a false positive.

There's a second, opposite way a test can mislead you: a **Type II error** — a false negative,
where a real effect exists but your test fails to detect it.

A clinical framing makes this concrete — think of it like a diagnostic test:

| | Disease actually present | Disease actually absent |
|---|---|---|
| **Test says positive** | Correct (true positive) | **Type I error** (false positive) |
| **Test says negative** | **Type II error** (false negative) | Correct (true negative) |

In hypothesis testing terms: Type I error = rejecting a true null hypothesis (α, usually set to
0.05). Type II error = failing to reject a false null hypothesis (β). **Power** (1 − β) is the
probability of correctly detecting a real effect when one exists.

### Exercise 3.3 — Seeing a Type II error happen

This time, we'll build in a **real** effect and see how often our test manages to detect it.


**🤖 AI Prompt to try:**

> "Explain the difference between a Type I error and a Type II error using a clinical diagnostic
> test as an example. Then explain what 'statistical power' means and what typically increases
> it."

Compare its explanation to the table above — does it match, or does it blur the two together?


In [ ]:
# 🔧 CHANGE THIS: try sample_size = 10, then 100 — watch how often the real effect gets detected
sample_size = 20
true_effect_size = 0.5  # a modest, realistic effect — not huge, not tiny

n_simulations = 1000
np.random.seed(0)
detected = 0

for _ in range(n_simulations):
    # This time the two groups genuinely differ by `true_effect_size`
    sample_a = np.random.normal(loc=0, scale=1, size=sample_size)
    sample_b = np.random.normal(loc=true_effect_size, scale=1, size=sample_size)
    _, p = stats.ttest_ind(sample_a, sample_b)
    if p < 0.05:
        detected += 1

power = detected / n_simulations
type_ii_error_rate = 1 - power

print(f"With n = {sample_size} per group:")
print(f"  Detected the real effect: {detected}/{n_simulations} times ({power:.1%} — this is the test's power)")
print(f"  Missed the real effect (Type II error): {type_ii_error_rate:.1%} of the time")


### 💬 Discuss

- Re-run the cell with `sample_size = 100`. What happens to the Type II error rate? Why?
- Notice that changing sample size did **not** change the Type I error rate in Exercise 3.2 (it's
  fixed by your chosen α, e.g. 0.05) — but it *does* change your Type II error rate / power. Why
  are these two error types affected so differently by sample size?
- A common, expensive mistake in biomedical research: running an underpowered study (too few
  patients, too few replicates), getting a non-significant result, and concluding "there's no
  effect." What's wrong with that conclusion, given what you just saw?


## Wrap-up — Day 1 reflection

Before we close, jot down (or discuss):

1. In your own words, what does a 95% confidence interval mean?
2. In your own words, what does a p-value mean — and what's one common way people misstate it?
3. In your own words, what's the difference between a Type I and a Type II error, and how does
   sample size affect each one differently?
4. When you asked an AI tool to compute a CI or p-value today, did its **explanation** hold up to
   scrutiny, or did you catch it oversimplifying? What did you do about it?

Tomorrow: we bring in real gene expression data and tackle the multiple-testing problem head-on.
